# Modèle 2 — Random Forest Regression
**Projet ML 2026 — UMONS | Groupe 3**

Random Forest est un algorithme d'ensemble qui combine plusieurs arbres de décision.
Chaque arbre est entraîné sur un sous-ensemble aléatoire des données et des features.
La prédiction finale est la moyenne de tous les arbres.

In [ ]:
# Imports et configuration
import sys
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import cross_val_score, KFold, GridSearchCV
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline

# Import des fonctions utilitaires du projet
sys.path.append("../src")
from utils import compute_rmse, print_cv_results, create_submission

pd.set_option('display.float_format', '{:.4f}'.format)

# Chargement des données finales (nettoyées et fusionnées dans 01_EDA.ipynb)
train = pd.read_csv("../data/train_final.csv")
test  = pd.read_csv("../data/test_final.csv")

print(f"Train : {train.shape}")
print(f"Test  : {test.shape}")

In [ ]:
# Séparation features / cible
y = train["Ja in Prozent"]

# One Hot Encoding sur Kanton
train_encoded = pd.get_dummies(train, columns=["Kanton"])
test_encoded  = pd.get_dummies(test,  columns=["Kanton"])

# Supprimer les colonnes inutiles
X = train_encoded.drop(columns=["Ja in Prozent", "Gemeinde", "commune_id", "Kantons-Nummer"])
X_test = test_encoded.drop(columns=["Gemeinde", "commune_id", "Kantons-Nummer"])

# Aligner les colonnes train et test
X_test = X_test.reindex(columns=X.columns, fill_value=0)

print(f" X train : {X.shape}")
print(f" X test  : {X_test.shape}")
print(f" y train : {y.shape}")

## Modèle Random Forest de base
On commence par un modèle de base avec les hyperparamètres par défaut
pour avoir une première idée des performances.

In [ ]:
# Modèle de base avec hyperparamètres par défaut
pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="mean")),
    ("rf", RandomForestRegressor(n_estimators=100, random_state=42))
])

# Cross-validation 5 folds
kf = KFold(n_splits=5, shuffle=True, random_state=42)
scores = cross_val_score(
    pipeline, X, y,
    cv=kf,
    scoring="neg_root_mean_squared_error"
)

rmse_scores = -scores
print("=== Résultats Cross-Validation (5 folds) ===")
print(f"RMSE par fold : {rmse_scores.round(4)}")
print(f"RMSE moyen    : {rmse_scores.mean():.4f}")
print(f"RMSE std      : {rmse_scores.std():.4f}")

##  Optimisation des hyperparamètres
On utilise GridSearchCV pour trouver la meilleure combinaison d'hyperparamètres.

In [ ]:
# Grille d'hyperparamètres à tester
param_grid = {
    "rf__n_estimators": [100, 200, 300],
    "rf__max_depth": [None, 10, 20, 30],
    "rf__min_samples_split": [2, 5, 10],
    "rf__max_features": ["sqrt", "log2", 0.5]
}

pipeline_gs = Pipeline([
    ("imputer", SimpleImputer(strategy="mean")),
    ("rf", RandomForestRegressor(random_state=42))
])

grid_search = GridSearchCV(
    pipeline_gs,
    param_grid,
    cv=kf,
    scoring="neg_root_mean_squared_error",
    n_jobs=-1,
    verbose=1
)

grid_search.fit(X, y)

print(f"\nMeilleurs hyperparamètres : {grid_search.best_params_}")
print(f"Meilleur RMSE CV : {-grid_search.best_score_:.4f}")

## Entraînement du modèle final
On entraîne le modèle Random Forest avec les meilleurs hyperparamètres trouvés.

In [ ]:
# Entraînement du modèle final
pipeline_final = Pipeline([
    ("imputer", SimpleImputer(strategy="mean")),
    ("rf", RandomForestRegressor(
        n_estimators=300,
        max_depth=20,
        max_features=0.5,
        min_samples_split=2,
        random_state=42
    ))
])

pipeline_final.fit(X, y)

y_pred_train = pipeline_final.predict(X)
rmse_train   = compute_rmse(y, y_pred_train)
best_rmse_v1 = -grid_search.best_score_

print(f"Modèle Random Forest v1 entraîné")
print(f"RMSE train complet    : {rmse_train:.4f}")
print(f"RMSE cross-validation : {best_rmse_v1:.4f}")

## Optimisation v2 - Correction de l'overfitting
Le modèle v1 montre de l'overfitting (RMSE train=2.27 vs RMSE CV=6.24).
On relance la recherche avec des hyperparamètres plus restrictifs.

In [ ]:
# Entraînement du modèle final v2
pipeline_final_v2 = Pipeline([
    ("imputer", SimpleImputer(strategy="mean")),
    ("rf", RandomForestRegressor(
        n_estimators=500,
        max_depth=15,
        max_features=0.3,
        min_samples_split=5,
        random_state=42
    ))
])

# Cross-validation pour estimer le RMSE réel du modèle v2
cv_scores = cross_val_score(
    pipeline_final_v2, X, y,
    cv=kf,
    scoring="neg_root_mean_squared_error"
)
best_rmse_v2 = -cv_scores.mean()

# Entraînement sur tout le dataset
pipeline_final_v2.fit(X, y)

# Evaluation sur le train complet
y_pred_train = pipeline_final_v2.predict(X)
rmse_train   = compute_rmse(y, y_pred_train)

print(f"Modèle Random Forest v2 entraîné")
print(f"RMSE train complet    : {rmse_train:.4f}")
print(f"RMSE cross-validation : {best_rmse_v2:.4f}")

## Prédictions et soumission Kaggle

In [ ]:
# Prédictions sur le test
y_pred_test = pipeline_final_v2.predict(X_test)

# Création du fichier de soumission
create_submission(
    commune_ids=test["commune_id"],
    y_pred=y_pred_test,
    filename="../submissions/submission_RF_v1.csv"
)